# 데이터 결합 — `concat` · `merge`

> ✏️ **실습본** — 마크다운 설명을 읽고, 빈칸(`# 👉`)을 직접 채워 실행하세요. 막히면 완성본을 참고하세요.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------
# 한글 폰트 설정 (OS 자동 감지)
# ---------------------------------------
import platform
_os = platform.system()
if _os == 'Windows':
    plt.rc('font', family='Malgun Gothic')     # 윈도우
elif _os == 'Darwin':
    plt.rc('font', family='AppleGothic')        # 맥
else:
    plt.rc('font', family='NanumGothic')        # 리눅스 / 구글 코랩
plt.rc('axes', unicode_minus=False)

---
## 1. `concat` — 이어 붙이기
같은 구조의 데이터를 위아래(행)로 잇습니다. `ignore_index=True`로 인덱스를 새로 매깁니다.

In [8]:
jan = pd.DataFrame({'월': '1월', '상품': ['아메리카노', '라떼'], '매출': [120, 95]})
feb = pd.DataFrame({'월': '2월', '상품': ['아메리카노', '라떼'], '매출': [135, 110]})
display(jan)
display(feb)

,월,상품,매출
0,1월,아메리카노,120
1,1월,라떼,95


,월,상품,매출
0,2월,아메리카노,135
1,2월,라떼,110


In [12]:
# 👉 여기에 코드를 작성하세요
sales = pd.concat([jan,feb], ignore_index=True)
sales

,월,상품,매출
0,1월,아메리카노,120
1,1월,라떼,95
2,2월,아메리카노,135
3,2월,라떼,110


> 💡 `ignore_index`는 인덱스만 정리합니다. **업무 키(주문번호 등)의 중복은 따로 확인**해야 합니다 — 실전에서 중요.

In [21]:
students = pd.DataFrame([{"학번":1, "이름":"철수", "나이":20},
                         {"학번":2, "이름":"영희", "나이":30},
                         {"학번":3, "이름":"길동", "나이":10}])
students = students.set_index("학번")
students

,이름,나이
학번,,
1,철수,20
2,영희,30
3,길동,10


In [22]:
scores = pd.DataFrame([{"학번":1, "평균점수":85},
                       {"학번":2, "평균점수":90}])
scores = scores.set_index("학번")
scores

,평균점수
학번,
1,85
2,90


In [23]:
pd.concat([students, scores], axis=1)

,이름,나이,평균점수
학번,,,
1,철수,20,85.0
2,영희,30,90.0
3,길동,10,NaN


---
## 2. `merge` — 공통 키로 연결
두 표를 공통 키의 값으로 짝지어 옆으로 붙입니다.

In [24]:
customers = pd.DataFrame({'customer_id':['C01','C02','C03','C04'], 
                          'region':['서울','부산','서울','대전']})
orders    = pd.DataFrame({'order_id':['O01','O02','O03','O04'], 
                          'customer_id':['C01','C02','C02','C05'],
                          'order_amount':[50000,32000,18000,41000]})
display(customers)
display(orders)

,customer_id,region
0,C01,서울
1,C02,부산
2,C03,서울
3,C04,대전


,order_id,customer_id,order_amount
0,O01,C01,50000
1,O02,C02,32000
2,O03,C02,18000
3,O04,C05,41000


### `how` 네 가지 — 어느 쪽 키를 남길까

In [30]:
# inner(기본): 양쪽에 다 있는 C01,C02
# 👉 여기에 코드를 작성하세요

pd.merge(customers, orders, how='inner')

,customer_id,region,order_id,order_amount
0,C01,서울,O01,50000
1,C02,부산,O02,32000
2,C02,부산,O03,18000


In [31]:
# left: 모든 고객 유지(주문없는 C03,C04 NaN)
# 👉 여기에 코드를 작성하세요

pd.merge(customers, orders, how='left')

,customer_id,region,order_id,order_amount
0,C01,서울,O01,50000.0
1,C02,부산,O02,32000.0
2,C02,부산,O03,18000.0
3,C03,서울,NaN,NaN
4,C04,대전,NaN,NaN


In [32]:
# right: 모든 주문 유지(고객 없는 C05 NaN)
# 👉 여기에 코드를 작성하세요

pd.merge(customers, orders, how='right')

,customer_id,region,order_id,order_amount
0,C01,서울,O01,50000
1,C02,부산,O02,32000
2,C02,부산,O03,18000
3,C05,NaN,O04,41000


In [33]:
# outer: 양쪽 전부
# 👉 여기에 코드를 작성하세요

pd.merge(customers, orders, how='outer')

,customer_id,region,order_id,order_amount
0,C01,서울,O01,50000.0
1,C02,부산,O02,32000.0
2,C02,부산,O03,18000.0
3,C03,서울,NaN,NaN
4,C04,대전,NaN,NaN
5,C05,NaN,O04,41000.0


| `how` | 남기는 키 | 쓰는 상황 |
|---|---|---|
| `inner` | 공통 키 | 양쪽 정보가 다 있는 것만 |
| `left` | 왼쪽 전부 | 기준 데이터에 정보 덧붙이기 (가장 흔함) |
| `right` | 오른쪽 전부 | 오른쪽 기준 유지 |
| `outer` | 양쪽 전부 | 누락·불일치 점검 |

### 키 지정 · 타입 함정
이름이 다르면 `left_on`/`right_on`, 여러 컬럼이면 `on=[...]`.       
그리고 **문자열 `'001'` 과 숫자 `1` 은 결합되지 않습니다** — 타입을 맞추세요.

In [ ]:
# 이름이 다른 키
# 👉 여기에 코드를 작성하세요

In [ ]:
# 고객id를 기준으로 merge
# 👉 여기에 코드를 작성하세요

## 3. 관계 검증

### `validate`
- 두 데이터를 결합하기 전에 데이터간의 관계를 확인합니다.

| 관계 | 뜻 | 예시 | `validate` |
|---|---|---|---|
| 1:1 | 왼쪽 하나에 오른쪽 하나 | 회원 ↔ 회원 등급 |`one_to_one`|
| 1:N | 왼쪽 하나에 오른쪽 여럿 | 회원 1명 ↔ 주문 여러 건 |`one_to_many`|
| N:1 | 왼쪽 여럿에 오른쪽 하나 | 주문 여러 건 ↔ 상품 1개 |`many_to_one`|
| N:M | 왼쪽 여럿에 오른쪽 여럿 | 보통은 없는 관계 |`many_to_many`|


`validate`에 관계를 적어두면, 실제 데이터가 그와 **다를 때 에러로 알려줍니다.**

---

### `indicator` — 매칭상태를 표시해주는 도구

`indicator=True`를 주면 결과에 `_merge` 열이 생겨, 각 행이 어디서 왔는지 알려줍니다.

- `both` : 양쪽 표에 모두 있음
- `left_only` : 왼쪽 표에만 있음
- `right_only` : 오른쪽 표에만 있음

주문엔 있는데 회원 명단엔 없는 회원처럼 **짝을 못 찾은 행(미매칭)** 을 골라낼 때 씁니다. 

In [44]:
customers = pd.DataFrame({'customer_id':['C01','C01','C02','C03','C04'], 
                          'region':['제주', '서울','부산','서울','대전']})
orders    = pd.DataFrame({'order_id':['O01','O02','O03','O04'], 
                          'cust_id':['C01','C02','C02','C05'],
                          'order_amount':[50000,32000,18000,41000]})
display(customers)
display(orders)

,customer_id,region
0,C01,제주
1,C01,서울
2,C02,부산
3,C03,서울
4,C04,대전


,order_id,cust_id,order_amount
0,O01,C01,50000
1,O02,C02,32000
2,O03,C02,18000
3,O04,C05,41000


In [ ]:
# 👉 여기에 코드를 작성하세요

pd.merge(customers, orders, left_on='customer_id', right_on='cust_id',
         validate='one_to_many')

`indicator=True`가 만든 `_merge` 열로 **어느 쪽에만 있는지**(`both`/`left_only`/`right_only`)를 진단합니다.

In [ ]:
# 👉 여기에 코드를 작성하세요

---
## 4. 행 단위(grain) 맞추기
**한 행의 단위(grain)** 가 다르면 
=>  한쪽을 `groupby`로 집계해 단위를 맞춘 뒤 붙입니다.

In [46]:
# 주문: 한 행 = 주문 1건
orders = pd.DataFrame({
    '주문ID': ['O1', 'O2', 'O3'],
    '고객':   ['김지은', '박민수', '이서연'],
    '주문일': ['2025-01-02', '2025-01-03', '2025-01-05'],
})

# 주문상품: 한 행 = 상품 1건  (한 주문에 여러 행이 생김)
order_items = pd.DataFrame({
    '주문ID': ['O1', 'O1', 'O2', 'O3'],
    '상품':   ['노트북', '마우스', '키보드', '모니터'],
    '금액':   [1200, 30, 50, 250],
})

display(orders)        # 그레인: 주문 단위
display(order_items)   # 그레인: 주문에 담긴 상품 1건(주문+상품 단위)


,주문ID,고객,주문일
0,O1,김지은,2025-01-02
1,O2,박민수,2025-01-03
2,O3,이서연,2025-01-05


,주문ID,상품,금액
0,O1,노트북,1200
1,O1,마우스,30
2,O2,키보드,50
3,O3,모니터,250


In [50]:
# ① 주문상품을 '주문 단위'로 집계 → 그레인을 '주문'으로 맞춤
# 👉 여기에 코드를 작성하세요

order_items2 = order_items.groupby('주문ID')['금액'].sum().to_frame().reset_index()
order_items2

,주문ID,금액
0,O1,1230
1,O2,50
2,O3,250


In [52]:
# ② 주문(1행) ↔ 집계(1행) 를 1:1 로 결합
# 👉 여기에 코드를 작성하세요

pd.merge(orders, order_items2, validate='one_to_one')

,주문ID,고객,주문일,금액
0,O1,김지은,2025-01-02,1230
1,O2,박민수,2025-01-03,50
2,O3,이서연,2025-01-05,250


> 💡 이 '집계 후 merge'가 여러 소스를 하나로 만드는 실무의 핵심 패턴입니다. **결합 실전**에서 주문상품·주문·상품·고객을 이렇게 잇습니다.

---
# ▶ 실전 — 전자상거래 데이터 결합 (8단계)
`concat`·`merge`·`validate`·`indicator`를 실전 데이터에 적용합니다.      
이 데이터에는 **의도적으로 심어둔 오류**가 있습니다.      
문법 실행보다 **'예상 vs 실제'를 비교하며 진단**하는 데 집중하세요.

> ⚠️ **ID는 문자열!** `C0001`·`P0001`처럼 앞자리 0이 있어 `dtype=str`로 읽지 않으면 `1`로 바뀝니다.

## 1단계 · 데이터 구조 확인

이번 실전은 한 **온라인 쇼핑몰**의 데이터입니다. 성격에 따라 5개 파일로 나뉘어 있고, 우리는 이걸 결합해 하나의 분석표로 만들 겁니다.

| 파일 | 한 행의 의미 | 주요 키 | 담긴 정보 |
|---|---|---|---|
| `customers` | 고객 1명 | `customer_id` | 이름·지역·등급·가입일·연령대 |
| `orders_2025H1` / `H2` | 주문 1건 (상·하반기) | `order_id` | 고객·주문일·상태·결제수단·주문금액 |
| `order_items` | 주문에 담긴 상품 1건 | `order_item_id` | 주문·상품·수량·판매단가 |
| `products` | 상품 정보 1건 (버전 이력 포함) | `product_id` | 상품명·카테고리·브랜드·기준가 |

**연결 키**     
주문↔고객은 `customer_id`      
주문↔주문상품은 `order_id`      
주문상품↔상품은 `product_id`로 이어집니다.

> ⚠️ **ID는 문자열로 읽습니다.** `C0001`·`P0001`처럼 앞자리 0이 있어, 숫자로 읽으면 `1`이 되어 버립니다 → `dtype=str`.

In [ ]:
# 데이터 불러오기
customers = pd.read_csv('data/customers.csv', dtype=str)
h1 = pd.read_csv('data/orders_2025H1.csv', dtype=str)
h2 = pd.read_csv('data/orders_2025H2.csv', dtype=str)
items = pd.read_csv('data/order_items.csv', dtype=str)
products = pd.read_csv('data/products.csv', dtype=str)

# ('데이터프레임이름', 데이터프레임)
dfs = [('고객:customers',customers), ('상반기주문:h1',h1), ('하반기주문:h2',h2), ('주문-상품:items',items), ('상품정보:products',products)]

In [ ]:
# 👉 여기에 코드를 작성하세요

In [ ]:
# 데이터프레임 정보 확인
# 👉 여기에 코드를 작성하세요

In [ ]:
# 자료형 변경
# 👉 여기에 코드를 작성하세요

# 👉 여기에 코드를 작성하세요

## 2단계 · 키 품질 점검
결합 전에 각 키의 (고유값·중복행·결측)을 표로 확인합니다.

### 잠깐 — '키(key)'가 뭔가요?
각 행을 구분하고, 다른 표와 연결할 때 짝을 맞추는 기준이 되는 컬럼입니다.
키는 두 가지 역할을 합니다.

- **구분용** — 그 표에서 한 행을 유일하게 식별 (예: `customers`의 `customer_id` = 고객 한 명, 중복 없어야 함)
- **연결용** — 다른 표를 가리켜 이어줌 (예: `order_items`의 `order_id`는 "이 상품 줄이 어느 주문인지"를 가리킴)

앞에서 `merge(on='customer_id')` 할 때 적었던 그 `customer_id`가 바로 키입니다.

그래서 결합하기 전에 **키가 믿을 만한지 — 고유한가(중복 없나), 빈 값은 없나 —** 를 먼저 확인합니다. 

In [ ]:
# 👉 여기에 코드를 작성하세요

# 👉 여기에 코드를 작성하세요

> 🔎 `products.product_id` 의 **중복행이 0이 아님**에 주목 (5단계에서 해결).

## 3단계 · 상·하반기 연결 (`concat`) + 중복 진단

In [ ]:
# 상/하반기 연결
# 👉 여기에 코드를 작성하세요

In [ ]:
# 중복 진단
# 👉 여기에 코드를 작성하세요

In [ ]:
# 중복 데이터 확인
# 👉 여기에 코드를 작성하세요

In [ ]:
# 하반기의 데이터만 남기기
# 👉 여기에 코드를 작성하세요

## 4단계 · 주문 ↔ 주문상품 결합 (`merge`, 1:N)

In [ ]:
# 👉 여기에 코드를 작성하세요

In [ ]:
# 👉 여기에 코드를 작성하세요

In [ ]:
# 👉 여기에 코드를 작성하세요

## 5단계 · 상품정보 결합
- 상품 이력 중복 정리
  - 같은 product_id 의 여러 버전 중 '최신'만 남기기 
  - products 에는 같은 상품이 수정 이력 때문에 여러 줄 있음

In [ ]:
# 중복행 확인 => 같은 상품이 리뉴얼 되어 중복 저장되어 있는 것으로 확인
# 👉 여기에 코드를 작성하세요

In [ ]:
# 최신 데이터만 남기기 : 최신순으로 정렬 -> 최신데이터만 남기고 중복 제거
# 👉 여기에 코드를 작성하세요

In [ ]:
# 주문상품 + 상품 결합
# 👉 여기에 코드를 작성하세요

In [ ]:
# 미매칭 데이터 진단 => 미등록 상품 8건
# 👉 여기에 코드를 작성하세요

In [ ]:
# 👉 여기에 코드를 작성하세요

In [ ]:
# 진단 끝났으니 보조 열(_merge) 제거
# 👉 여기에 코드를 작성하세요

## 6단계 · 고객정보 결합

In [ ]:
# 고객 정보 결합
# 👉 여기에 코드를 작성하세요

In [ ]:
# 미매칭 데이터 확인
# 👉 여기에 코드를 작성하세요

In [ ]:
# 👉 여기에 코드를 작성하세요

## 7단계 · 통합 데이터 완성 & 저장
매출·월 변수를 만들고 `통합주문데이터.csv`로 저장합니다. **이 파일은 다음 파트(구조 변환)에서 이어 씁니다.**

In [ ]:
# 👉 여기에 코드를 작성하세요

## 8단계 · 최종 검증

결합이 다 끝났으니, **완성된 데이터가 믿을 만한지** 네 가지로 점검합니다. 하나라도 이상하면 저장·분석으로 넘어가기 전에 원인을 확인해야 합니다.

| 확인 항목 | 정상이면 | 무엇을 잡아내나 |
|---|---|---|
| 행 수 유지 | `True` | 결합으로 행이 늘지 않음(뻥튀기 없음) |
| 키 고유 | `True` | 주문상품 키가 여전히 고유 |
| 미매칭 비율 | 아주 작음 | 짝 못 찾은 데이터가 예상 범위인지 |
| 매출 교차검증 | `100%` | 주문상품 매출 합이 주문 총액과 일치 → 결합이 정확 |

특히 마지막 **교차검증**이 핵심입니다 — 한 주문의 `item_sales` 합계가 그 주문의 `order_total`과 같아야 하므로, 이게 맞으면 "주문상품이 올바른 주문에 정확히 붙었다"는 강력한 증거가 됩니다.

In [ ]:
# 👉 여기에 코드를 작성하세요
# 교차검증 — 주문상품 매출 합계 == 주문 order_total
# 👉 여기에 코드를 작성하세요

---
## 📋 산출물 ① 결합 검증표
| 단계 | 왼쪽 | 오른쪽 | 키 | 관계 | 결합후 행수 | 미매칭 | 처리 |
|---|---|---|---|---|---:|---:|---|
| 1 | 주문 | 주문상품 | order_id | 1:N | 12,500 | 0 | 정상 |
| 2 | +상품 | 상품(최신) | product_id | N:1 | 12,500 | 8 | 미등록 목록 기록 |
| 3 | +고객 | 고객 | customer_id | N:1 | 12,500 | 11 | 미등록 목록 기록 |

## 📝 산출물 ② 결합 진단 문장 3개
> 형식: 무엇을 기준으로 결합 → 무엇을 확인 → 어떤 문제 → 어떻게 처리

1. 주문 기준으로 주문상품을 order_id로 1:N 결합했고, 행 수(12,500)와 키 고유성이 유지되어 비정상 증가는 없었다.
2. 상품 결합에서 8행이 미등록(P9001~)으로 확인돼, 삭제 대신 목록으로 남기고 기준정보 확인 대상으로 분류했다.
3. 상품 product_id가 이력으로 중복되어 N:1 검증이 실패했고, updated_at 최신 기준으로 정리해 행 뻥튀기를 막았다.